# 🌿 Plant Seedlings Classification – CNN

**Dataset:** 4,750 images (128×128 px, RGB) across **12 plant species**  
**Goal:** Train a Convolutional Neural Network (CNN) to classify plant seedlings by species.  

---
### Pipeline Overview
1. Import libraries
2. Load data (`images_plant.npy` + `Labels_plant.csv`)
3. Exploratory Data Analysis (EDA)
4. Pre-processing (BGR→RGB, resize 128→64, normalise)
5. Train/Val/Test split + label encoding
6. **Model 1** – Baseline CNN
7. **Model 2** – CNN + Data Augmentation + LR Scheduler
8. Evaluation & comparison

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                     Dropout, BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

print("TensorFlow version:", tf.__version__)
print("NumPy version     :", np.__version__)
print("Libraries imported successfully!")

## 2. Load Data

In [ ]:
# Load images and labels
images = np.load('images_plant.npy')
labels = pd.read_csv('Labels_plant.csv')

print("Images loaded  :", images.shape)
print("Labels loaded  :", labels.shape)
print("\nFirst 5 labels:")
print(labels.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape of images array :", images.shape)
print("Shape of labels dataframe:", labels.shape)
print("\nImage dimensions: {}x{} pixels, {} colour channels".format(
    images.shape[1], images.shape[2], images.shape[3]))
print("\nData type of images:", images.dtype)
print("Pixel value range  : [{}, {}]".format(images.min(), images.max()))
print("\nClass distribution:")
print(labels['Label'].value_counts())
print("\nTotal classes:", labels['Label'].nunique())

In [ ]:
species = labels['Label'].unique()
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle("Sample Images from Each Plant Species", fontsize=16, fontweight='bold')

for i, sp in enumerate(sorted(species)):
    ax = axes[i//4][i%4]
    idx = labels[labels['Label'] == sp].index[0]
    # Convert BGR to RGB for display
    img_rgb = cv2.cvtColor(images[idx], cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(sp, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 5))
species_counts = labels['Label'].value_counts().sort_values(ascending=False)
bars = plt.bar(species_counts.index, species_counts.values,
               color=plt.cm.Set2(np.linspace(0,1,12)), edgecolor='black', linewidth=0.7)
plt.xticks(rotation=35, ha='right', fontsize=10)
plt.ylabel("Count", fontsize=12)
plt.title("Class Distribution of Plant Species", fontsize=14, fontweight='bold')
for bar, cnt in zip(bars, species_counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+4, str(cnt),
             ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.ylim(0, 750)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Pre-processing

Steps:
- Convert BGR → RGB (OpenCV loads images in BGR order)
- Resize 128×128 → 64×64 to reduce computation
- Normalise pixel values to [0, 1]

In [ ]:
# OpenCV loads images in BGR format; convert to RGB for correct colour display
images_rgb = np.array([cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in images])
print("Shape after BGR\u2192RGB conversion:", images_rgb.shape)
print("Conversion complete \u2013 colours are now in correct RGB order.")

In [ ]:
# Resize all images from 128x128 to 64x64
IMG_SIZE = 64
images_resized = np.array([cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                            for img in images_rgb], dtype=np.uint8)
print("Original shape :", images_rgb.shape)
print("Resized shape  :", images_resized.shape)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle("Images Before (BGR 128\u00d7128) and After Pre-processing (RGB 64\u00d764)",
             fontsize=12, fontweight='bold')

sample_species = sorted(labels['Label'].unique())[:6]
for i, sp in enumerate(sample_species):
    idx = labels[labels['Label'] == sp].index[0]
    axes[0, i].imshow(cv2.cvtColor(images[idx], cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(sp[:13], fontsize=8); axes[0, i].axis('off')
    axes[1, i].imshow(images_resized[idx])
    axes[1, i].set_title(sp[:13], fontsize=8); axes[1, i].axis('off')

axes[0, 0].set_ylabel("Before\n(BGR 128\u00d7128)", fontsize=9, fontweight='bold')
axes[1, 0].set_ylabel("After\n(RGB 64\u00d764)", fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Train / Validation / Test Split & Label Encoding

- **80%** train, **20%** test  
- **15%** of training set → validation  
- Labels are **LabelEncoded** then **one-hot encoded** for categorical cross-entropy

In [ ]:
# Split: 80% train, 20% test; then 15% of train → validation
le = LabelEncoder()
y_encoded = le.fit_transform(labels['Label'].values)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    images_resized, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_train, y_train_raw, test_size=0.15, random_state=42, stratify=y_train_raw)

print("Training set   :", X_train.shape)
print("Validation set :", X_val.shape)
print("Test set       :", X_test.shape)

In [ ]:
# One-hot encode the labels for categorical cross-entropy
NUM_CLASSES = 12
y_train = to_categorical(y_train_raw, NUM_CLASSES)
y_val   = to_categorical(y_val_raw,   NUM_CLASSES)
y_test  = to_categorical(y_test_raw,  NUM_CLASSES)

print("Encoded classes:", list(le.classes_))
print("\ny_train shape:", y_train.shape)
print("y_val   shape:", y_val.shape)
print("y_test  shape:", y_test.shape)

In [ ]:
# Scale pixel values to [0, 1]
X_train_n = X_train.astype('float32') / 255.0
X_val_n   = X_val.astype('float32')   / 255.0
X_test_n  = X_test.astype('float32')  / 255.0

print("Pixel range after normalisation:")
print("  Train \u2013 min: {:.4f}, max: {:.4f}".format(X_train_n.min(), X_train_n.max()))
print("  Val   \u2013 min: {:.4f}, max: {:.4f}".format(X_val_n.min(),   X_val_n.max()))
print("  Test  \u2013 min: {:.4f}, max: {:.4f}".format(X_test_n.min(),  X_test_n.max()))

## 6. Model 1 – Baseline CNN

Architecture: **3 convolutional blocks** (Conv2D → BatchNorm → MaxPool → Dropout) followed by a **Dense classifier**.

| Block | Filters | Kernel | Dropout |
|-------|---------|--------|---------|
| 1     | 32      | 3×3    | 0.25    |
| 2     | 64      | 3×3    | 0.25    |
| 3     | 128     | 3×3    | 0.40    |
| FC    | 256     | —      | 0.50    |

In [ ]:
def build_model(input_shape=(64, 64, 3), num_classes=12):
    model = Sequential([
        # Block 1
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),
        # Block 2
        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),
        # Block 3
        Conv2D(128, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.40),
        # Fully connected
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.50),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model1 = build_model()
model1.summary()

In [ ]:
EPOCHS = 30
BATCH_SIZE = 32

history1 = model1.fit(
    X_train_n, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_n, y_val),
    verbose=1
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS+1)

ax1.plot(epochs_range, history1.history['accuracy'], 'b-o', ms=4, label='Train')
ax1.plot(epochs_range, history1.history['val_accuracy'], 'r-s', ms=4, label='Validation')
ax1.set_title("Model 1 \u2013 Accuracy", fontweight='bold')
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(alpha=0.4)

ax2.plot(epochs_range, history1.history['loss'], 'b-o', ms=4, label='Train')
ax2.plot(epochs_range, history1.history['val_loss'], 'r-s', ms=4, label='Validation')
ax2.set_title("Model 1 \u2013 Loss", fontweight='bold')
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(alpha=0.4)

plt.suptitle("Model 1 Training History (Baseline CNN)", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Evaluate on test set
test_loss1, test_acc1 = model1.evaluate(X_test_n, y_test, verbose=0)
print(f"Model 1 \u2013 Test Loss    : {test_loss1:.4f}")
print(f"Model 1 \u2013 Test Accuracy: {test_acc1:.4f} ({test_acc1*100:.2f}%)")

# Predictions
y_pred1 = np.argmax(model1.predict(X_test_n), axis=1)

print("\nClassification Report \u2013 Model 1:")
print(classification_report(y_test_raw, y_pred1, target_names=le.classes_))

In [ ]:
cm1 = confusion_matrix(y_test_raw, y_pred1)
plt.figure(figsize=(11, 9))
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues',
            xticklabels=[s[:12] for s in le.classes_],
            yticklabels=[s[:12] for s in le.classes_],
            linewidths=0.5)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Confusion Matrix \u2013 Model 1 (Baseline CNN)", fontsize=13, fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

## 7. Model 2 – CNN + Data Augmentation + LR Scheduler

Improvements over Model 1:
- **Data Augmentation**: rotation, shift, shear, zoom, horizontal flip
- **ReduceLROnPlateau**: halves LR when `val_loss` stalls for 4 epochs
- **EarlyStopping**: stops training if no improvement for 8 epochs

In [ ]:
# ── Learning Rate Scheduler ──────────────────────────────────────────────────
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                               patience=4, verbose=1, min_lr=1e-6)
early_stop  = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

# ── Data Augmentation ─────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# No augmentation for validation / test
val_datagen = ImageDataGenerator()

train_gen = train_datagen.flow(X_train_n, y_train, batch_size=BATCH_SIZE, seed=42)
val_gen   = val_datagen.flow(X_val_n, y_val, batch_size=BATCH_SIZE, shuffle=False)

print("Data augmentation configured successfully.")
print("Training generator \u2013 steps per epoch:", len(X_train_n) // BATCH_SIZE)

In [ ]:
# Build Model 2 (same architecture, trained with augmented data + LR schedule)
model2 = build_model()

history2 = model2.fit(
    train_gen,
    steps_per_epoch=len(X_train_n) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_gen,
    validation_steps=len(X_val_n) // BATCH_SIZE,
    callbacks=[lr_reducer, early_stop],
    verbose=1
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range2 = range(1, len(history2.history['accuracy'])+1)

ax1.plot(epochs_range2, history2.history['accuracy'], 'b-o', ms=4, label='Train')
ax1.plot(epochs_range2, history2.history['val_accuracy'], 'r-s', ms=4, label='Validation')
ax1.set_title("Model 2 \u2013 Accuracy", fontweight='bold')
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(alpha=0.4)

ax2.plot(epochs_range2, history2.history['loss'], 'b-o', ms=4, label='Train')
ax2.plot(epochs_range2, history2.history['val_loss'], 'r-s', ms=4, label='Validation')
ax2.set_title("Model 2 \u2013 Loss", fontweight='bold')
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(alpha=0.4)

plt.suptitle("Model 2 Training History (CNN + Data Augmentation + LR Scheduler)",
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
test_loss2, test_acc2 = model2.evaluate(X_test_n, y_test, verbose=0)
print(f"Model 2 \u2013 Test Loss    : {test_loss2:.4f}")
print(f"Model 2 \u2013 Test Accuracy: {test_acc2:.4f} ({test_acc2*100:.2f}%)")

y_pred2 = np.argmax(model2.predict(X_test_n), axis=1)

print("\nClassification Report \u2013 Model 2:")
print(classification_report(y_test_raw, y_pred2, target_names=le.classes_))

In [ ]:
cm2 = confusion_matrix(y_test_raw, y_pred2)
plt.figure(figsize=(11, 9))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens',
            xticklabels=[s[:12] for s in le.classes_],
            yticklabels=[s[:12] for s in le.classes_],
            linewidths=0.5)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Confusion Matrix \u2013 Model 2 (CNN + Data Augmentation)",
          fontsize=13, fontweight='bold')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

## 8. Sample Predictions – Final Model (Model 2)

Green title = correct prediction | Red title = misclassified

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle("Sample Predictions \u2013 Final Model (Model 2)", fontsize=14, fontweight='bold')

for i in range(12):
    ax = axes[i//4][i%4]
    # Get one test sample from each class
    idx = np.where(y_test_raw == i)[0][0]
    pred_class = y_pred2[idx]
    colour = 'green' if pred_class == i else 'red'
    ax.imshow(X_test_n[idx])
    ax.set_title(f"True: {le.classes_[i][:14]}\nPred: {le.classes_[pred_class][:14]}",
                 fontsize=7.5, color=colour, fontweight='bold')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(colour); spine.set_linewidth(3)

plt.tight_layout(); plt.show()

## Summary

| Model | Description | Notes |
|-------|-------------|-------|
| Model 1 | Baseline CNN (3 blocks) | Fixed LR, no augmentation |
| Model 2 | CNN + Augmentation + LR Scheduler | EarlyStopping, ReduceLROnPlateau |

Model 2 is expected to generalise better due to data augmentation reducing overfitting and the LR scheduler enabling finer convergence.